In [ ]:
import soundfile
import torch
import matplotlib.pyplot as plt


y, sr = soundfile.read("/home/vklimkov/workspace/vllm/vllm/fastconformer_test_data/sample.wav", dtype="float32", always_2d=True)
assert sr == 16000
test_audio = torch.from_numpy(y.T).contiguous().cuda()[0]  # shape [samples]
# we have x1280 downsampling, so if we want 20 frames, we need:
#samples_num = 90 * 160 * 8
#print(test_audio.shape, test_audio.dtype)
#test_audio = test_audio[:samples_num]
print(test_audio.shape, test_audio.dtype)
plt.plot(test_audio.cpu().numpy())
plt.show()

In [ ]:
import torch

from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
from vllm.sampling_params import SamplingParams

engine_args = AsyncEngineArgs(
    model="./fastconformer_model/",
    max_model_len=1024,
    gpu_memory_utilization=0.8,
    block_size=128,
    skip_tokenizer_init=True,
    enable_prefix_caching=False,
    dtype="float32",
    #enforce_eager=True,
    #load_format="dummy",
    compilation_config={"cudagraph_mode": "FULL"},
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=4096, skip_sampling=True)


# reshape audio to [target_t, factor]
audio = test_audio.cpu()
# run preemphasis here
audio = torch.cat(
    (audio[0:1], audio[1:] - 0.97 * audio[:-1]), dim=0
)
print(audio.shape)

request_id = "1"
step = 2  # number of frames per decode step (must match num_output_tokens_per_step in config)
prompt_len = step
i = 0
FRAME_LEN = 1280  # 16000 * 0.08
inputs = {
    # dummy tokens
    "prompt_token_ids": [0] * prompt_len,
    # actual inpust to the model in prefill stage
    "custom_inputs": {
        "audio": audio[i * FRAME_LEN:(i+prompt_len) * FRAME_LEN].view(prompt_len, FRAME_LEN)  # 1 x 8*160
    }
}
i += prompt_len
outputs = []
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id=request_id):
    acoustic_emb = output.outputs[0].custom_outputs["acoustic_emb"].clone()
    outputs.append(acoustic_emb)
    if i >= audio.shape[0] // FRAME_LEN:
        break
    # prepare the next one
    inputs = {"audio": audio[i * FRAME_LEN:(i+step) * FRAME_LEN].view(step, FRAME_LEN)}
    i += step
    await engine.append_request(request_id=request_id, custom_inputs=inputs)

pred = torch.cat(outputs, dim=0).numpy()
print(pred.shape)


In [ ]:
import numpy as np
arr = np.load("fastconformer_test_data/nemo_outputs.npy").T
plt.imshow(pred.T, aspect="auto")
plt.colorbar()
plt.show()

plt.imshow(arr.T, aspect="auto")
plt.colorbar()
plt.show()

plt.imshow(pred.T - arr.T, aspect="auto")
plt.colorbar()
plt.show()

print(np.linalg.norm(pred - arr))